In [1]:
from huggingface_hub import hf_hub_download, list_repo_files
import tarfile
import zstandard as zstd
import io
import os

os.makedirs("unlab_atc_clips", exist_ok=True)

all_files = list(list_repo_files("nyuuzyou/liveatc", repo_type="dataset"))
archives = [f for f in all_files if f.endswith(".tar.zst")]

print(f"Found {len(archives)} archives")
print(archives[:5])

Found 213 archives
['data/06c.tar.zst', 'data/0v4.tar.zst', 'data/10c.tar.zst', 'data/1b9.tar.zst', 'data/1c9.tar.zst']


In [6]:
from huggingface_hub import list_repo_files
import os

os.makedirs("unlab_atc_clips", exist_ok=True)


def process_clip(mp3_bytes, silence_db=-40, padding_ms=100):
    seg = AudioSegment.from_file(io.BytesIO(mp3_bytes), format="mp3")
    seg = seg.set_channels(1)  # mono
    sr = seg.frame_rate

    samples = np.array(seg.get_array_of_samples(), dtype=np.float32)
    samples /= np.iinfo(seg.array_type).max  # normalize to [-1, 1]

    frame_len = 2048
    hop_len = 512
    pad_needed = (frame_len - len(samples) % frame_len) % frame_len
    padded = np.pad(samples, (0, pad_needed))

    n_frames = (len(samples) - frame_len) // hop_len + 1
    rms = np.array(
        [
            np.sqrt(np.mean(samples[i * hop_len : i * hop_len + frame_len] ** 2))
            for i in range(n_frames)
        ]
    )

    peak_rms = rms.max() if rms.max() > 0 else 1.0
    threshold = peak_rms * (10 ** (silence_db / 20))

    is_speech = rms > threshold
    intervals = []
    in_speech = False
    for i, speech in enumerate(is_speech):
        if speech and not in_speech:
            start = i * hop_len
            in_speech = True
        elif not speech and in_speech:
            end = min(i * hop_len + frame_len, len(samples))
            intervals.append((start, end))
            in_speech = False
    if in_speech:
        intervals.append((start, len(samples)))

    if not intervals:
        return None  # fully silent, skip

    pad_samples = np.zeros(int(sr * padding_ms / 1000), dtype=np.float32)
    segments = []
    for start, end in intervals:
        segments.append(samples[start:end])
        segments.append(pad_samples)

    processed = np.concatenate(segments[:-1])  # drop trailing pad

    buf = io.BytesIO()
    sf.write(buf, processed, sr, format="WAV")
    buf.seek(0)
    return buf.read()


all_files = list(list_repo_files("nyuuzyou/liveatc", repo_type="dataset"))
archives = [f for f in all_files if f.endswith(".tar.zst")]
print(f"Found {len(archives)} archives")

collected = 0
target = 200

for archive_name in archives:
    if collected >= target:
        break

    print(f"\nFetching {archive_name}...")
    local_path = hf_hub_download(
        repo_id="nyuuzyou/liveatc", filename=archive_name, repo_type="dataset"
    )

    with open(local_path, "rb") as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressed:
            with tarfile.open(fileobj=decompressed, mode="r|") as tar:
                for member in tar:
                    if collected >= target:
                        break
                    if not member.name.endswith(".mp3"):
                        continue
                    f = tar.extractfile(member)
                    if not f:
                        continue

                    mp3_bytes = f.read()
                    wav_bytes = process_clip(mp3_bytes, silence_db=-40, padding_ms=0)

                    if wav_bytes is None:
                        print(f"  [SKIP] {member.name} — fully silent")
                        continue

                    out_name = (
                        os.path.splitext(os.path.basename(member.name))[0] + ".wav"
                    )
                    with open(f"unlab_atc_clips/{out_name}", "wb") as out:
                        out.write(wav_bytes)

                    collected += 1
                    print(f"  [{collected}/{target}] {out_name}")

print(f"\nDone! {collected} processed clips saved to unlab_atc_clips/")

Found 213 archives

Fetching data/06c.tar.zst...
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-0830Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-2030Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-0030Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1230Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1100Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1900Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1930Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-2100Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1800Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1030Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1530Z.mp3 — fully silent
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-0400Z.mp3 — fully silent
  [

In [7]:
from huggingface_hub import hf_hub_download, list_repo_files
import tarfile
import zstandard as zstd
import os
import glob
import numpy as np
import soundfile as sf
from pydub import AudioSegment

os.makedirs("proc_unlab_atc_clips", exist_ok=True)


print("\nChunking into 30s segments...")

chunk_duration = 30
saved = 0

for wav_path in sorted(glob.glob("unlab_atc_clips/*.wav")):
    audio, sr = sf.read(wav_path)
    chunk_samples = chunk_duration * sr
    total_samples = len(audio)
    base = os.path.splitext(os.path.basename(wav_path))[0]

    if total_samples <= chunk_samples:
        sf.write(f"proc_unlab_atc_clips/{base}_chunk00.wav", audio, sr)
        saved += 1
    else:
        n_chunks = total_samples // chunk_samples
        for i in range(n_chunks):
            chunk = audio[i * chunk_samples : (i + 1) * chunk_samples]
            sf.write(f"proc_unlab_atc_clips/{base}_chunk{i:02d}.wav", chunk, sr)
            saved += 1

        remainder = audio[n_chunks * chunk_samples :]
        if len(remainder) >= 5 * sr:
            sf.write(
                f"proc_unlab_atc_clips/{base}_chunk{n_chunks:02d}.wav", remainder, sr
            )
            saved += 1

    print(
        f"  {base} → {max(1, total_samples // chunk_samples + (1 if total_samples % chunk_samples >= 5 * sr else 0))} chunk(s)"
    )

print(f"\nDone! {saved} chunks saved to proc_unlab_atc_clips/")


Chunking into 30s segments...
  10C-C81-3CK-CTAF-Aug-26-2024-0000Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0030Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0100Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0130Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0200Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0230Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0300Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0330Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0400Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0430Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0500Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0530Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0600Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0630Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0700Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0730Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0800Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0830Z → 64 chunk(s)
  10C-C81-3CK-CTAF-Aug-26-2024-0900Z → 64 chunk(s)


In [1]:
from huggingface_hub import hf_hub_download, list_repo_files
import tarfile
import zstandard as zstd
import os
import soundfile as sf
import subprocess

os.makedirs("liveatc_16k", exist_ok=True)

all_files = list(list_repo_files("nyuuzyou/liveatc", repo_type="dataset"))
archives = [f for f in all_files if f.endswith(".tar.zst")]

collected_seconds = 0
target_seconds = 50 * 3600
skipped = 0

MIN_RMS = 0.001
MIN_SPEECH_RATIO = 0.05


def decode_mp3_safe(mp3_bytes, target_sr=16000):
    result = subprocess.run(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            "pipe:0",
            "-af",
            "aresample=async=1000",
            "-ar",
            str(target_sr),
            "-ac",
            "1",
            "-f",
            "wav",
            "pipe:1",
        ],
        input=mp3_bytes,
        capture_output=True,
    )
    if not result.stdout:
        return None, target_sr
    buf = io.BytesIO(result.stdout)
    samples, sr = sf.read(buf, dtype="float32")
    return samples, sr


def has_enough_sound(samples, sr, min_rms=MIN_RMS, min_speech_ratio=MIN_SPEECH_RATIO):
    if len(samples) == 0:
        return False
    overall_rms = np.sqrt(np.mean(samples**2))
    if overall_rms < min_rms:
        return False
    frame_len = 1024
    n_frames = len(samples) // frame_len
    if n_frames == 0:
        return False
    frames = samples[: n_frames * frame_len].reshape(n_frames, frame_len)
    frame_rms = np.sqrt(np.mean(frames**2, axis=1))
    speech_ratio = np.mean(frame_rms > min_rms)
    return speech_ratio >= min_speech_ratio


for archive_name in archives:
    if collected_seconds >= target_seconds:
        break

    print(f"Fetching {archive_name}...")
    local_path = hf_hub_download(
        repo_id="nyuuzyou/liveatc", filename=archive_name, repo_type="dataset"
    )

    with open(local_path, "rb") as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressed:
            with tarfile.open(fileobj=decompressed, mode="r|") as tar:
                for member in tar:
                    if collected_seconds >= target_seconds:
                        break
                    if not member.name.endswith(".mp3"):
                        continue
                    f = tar.extractfile(member)
                    if not f:
                        continue

                    mp3_bytes = f.read()
                    samples, sr = decode_mp3_safe(mp3_bytes, target_sr=16000)

                    if samples is None or len(samples) == 0:
                        skipped += 1
                        print(
                            f"  [SKIP] {member.name} — failed to decode (skipped: {skipped})"
                        )
                        continue

                    if not has_enough_sound(samples, sr):
                        skipped += 1
                        print(
                            f"  [SKIP] {member.name} — silent or too quiet (skipped: {skipped})"
                        )
                        continue

                    out_name = (
                        os.path.splitext(os.path.basename(member.name))[0] + ".wav"
                    )
                    out_path = f"liveatc_16k/{out_name}"
                    sf.write(out_path, samples, sr, format="WAV")

                    duration = len(samples) / sr
                    collected_seconds += duration
                    print(
                        f"  [{collected_seconds / 3600:.2f}h / {target_seconds / 3600:.0f}h] {out_name}"
                    )

print(
    f"\nDone! {collected_seconds / 3600:.2f} hours saved to liveatc_16k/  |  {skipped} files skipped"
)

Fetching data/06c.tar.zst...
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-0830Z.mp3 — silent or too quiet (skipped: 1)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-2030Z.mp3 — silent or too quiet (skipped: 2)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-0030Z.mp3 — silent or too quiet (skipped: 3)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1230Z.mp3 — silent or too quiet (skipped: 4)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1100Z.mp3 — silent or too quiet (skipped: 5)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1900Z.mp3 — silent or too quiet (skipped: 6)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1930Z.mp3 — silent or too quiet (skipped: 7)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-2100Z.mp3 — silent or too quiet (skipped: 8)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1800Z.mp3 — silent or too quiet (skipped: 9)
  [SKIP] 06c/kord1n2_06c_ctaf/KORD1N2-06C-Aug-26-2024-1030Z.mp3 — silent or too 

In [3]:
print(all_files)

['.gitattributes', 'README.md', 'data/06c.tar.zst', 'data/0v4.tar.zst', 'data/10c.tar.zst', 'data/1b9.tar.zst', 'data/1c9.tar.zst', 'data/1g0.tar.zst', 'data/26ma.tar.zst', 'data/2b1.tar.zst', 'data/2g9.tar.zst', 'data/2gc.tar.zst', 'data/2j6.tar.zst', 'data/2nc0.tar.zst', 'data/2o3.tar.zst', 'data/2w2.tar.zst', 'data/39n.tar.zst', 'data/3b0.tar.zst', 'data/3n6.tar.zst', 'data/3r9.tar.zst', 'data/44n.tar.zst', 'data/46u.tar.zst', 'data/47n.tar.zst', 'data/4ny8.tar.zst', 'data/4s1.tar.zst', 'data/54az.tar.zst', 'data/5a2.tar.zst', 'data/5c1.tar.zst', 'data/7b2.tar.zst', 'data/7fl6.tar.zst', 'data/7w4.tar.zst', 'data/9s5.tar.zst', 'data/b18.tar.zst', 'data/c29.tar.zst', 'data/cnp3.tar.zst', 'data/csb3.tar.zst', 'data/csr3.tar.zst', 'data/cyam.tar.zst', 'data/cyav.tar.zst', 'data/cyaw.tar.zst', 'data/cybw.tar.zst', 'data/cycg.tar.zst', 'data/cyeg.tar.zst', 'data/cyfb.tar.zst', 'data/cyfc.tar.zst', 'data/cyfd.tar.zst', 'data/cyhc.tar.zst', 'data/cyhu.tar.zst', 'data/cyhz.tar.zst', 'data/cy

In [1]:
import os
import soundfile as sf


def process_atc_file(filepath, silence_db=-40, padding_sec=2.0):
    samples, sr = sf.read(filepath, dtype="float32")
    if samples.ndim > 1:
        samples = samples.mean(axis=1)

    frame_len = 1024
    hop_len = 256
    n_frames = (len(samples) - frame_len) // hop_len + 1

    if n_frames <= 0:
        return None, sr

    rms = np.array(
        [
            np.sqrt(np.mean(samples[i * hop_len : i * hop_len + frame_len] ** 2))
            for i in range(n_frames)
        ]
    )

    threshold = 10 ** (silence_db / 20)

    is_speech = rms > threshold
    intervals = []
    in_speech = False
    for i, speech in enumerate(is_speech):
        if speech and not in_speech:
            start = i * hop_len
            in_speech = True
        elif not speech and in_speech:
            end = min(i * hop_len + frame_len, len(samples))
            intervals.append((start, end))
            in_speech = False
    if in_speech:
        intervals.append((start, len(samples)))

    if not intervals:
        return None, sr

    merge_gap = int(0.5 * sr)
    merged = [list(intervals[0])]
    for start, end in intervals[1:]:
        if start - merged[-1][1] <= merge_gap:
            merged[-1][1] = end
        else:
            merged.append([start, end])

    pad = np.zeros(int(padding_sec * sr), dtype=np.float32)
    segments = [pad]
    for start, end in merged:
        segments.append(samples[start:end])
        segments.append(pad)

    result = np.concatenate(segments)
    return result, sr


def process_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.endswith(".wav")]

    skipped = 0
    for fname in files:
        in_path = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)

        result, sr = process_atc_file(in_path)
        if result is None:
            print(f"  [SKIP] {fname} — no audio above threshold")
            skipped += 1
            continue

        sf.write(out_path, result, sr, format="WAV")
        print(f"  [OK] {fname} — {len(result) / sr:.1f}s")

    print(f"\nDone. {len(files) - skipped} processed, {skipped} skipped.")


process_directory("liveatc_16k", "liveatc_processed")

  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-2030Z.wav — 4.3s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-0430Z.wav — 6.4s
  [OK] 1B9-Aug-26-2024-1400Z.wav — 121.3s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1630Z.wav — 71.5s
  [OK] 1B9-2-CTAF-Aug-26-2024-1500Z.wav — 172.0s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-0830Z.wav — 6.3s
  [SKIP] 1G0-CTAF-Aug-26-2024-0730Z.wav — no audio above threshold
  [OK] K0V4-CTAF-Aug-26-2024-1930Z.wav — 265.3s
  [OK] K0V4-CTAF-Aug-26-2024-1530Z.wav — 226.4s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-0200Z.wav — 6.4s
  [OK] 1B9-UEC-CTAF-Aug-26-2024-1530Z.wav — 136.8s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1000Z.wav — 4.1s
  [OK] 1C9-CTAF-Aug-26-2024-0200Z.wav — 130.7s
  [OK] 1B9-2-CTAF-Aug-26-2024-1330Z.wav — 114.3s
  [OK] K0V4-CTAF-Aug-26-2024-1730Z.wav — 140.4s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1200Z.wav — 36.0s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-0000Z.wav — 92.3s
  [OK] K0V4-CTAF-Aug-26-2024-1100Z.wav — 175.9s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-2230Z.wav — 82.5s
  [SKIP] 1G0-CTAF-Aug-

In [13]:
import os
import soundfile as sf


def process_atc_file(filepath, silence_db=-20, padding_sec=1.0, min_speech_sec=2.0):
    samples, sr = sf.read(filepath, dtype="float32")
    if samples.ndim > 1:
        samples = samples.mean(axis=1)

    frame_len = 1024
    hop_len = 256
    n_frames = (len(samples) - frame_len) // hop_len + 1

    if n_frames <= 0:
        return None, sr

    rms = np.array(
        [
            np.sqrt(np.mean(samples[i * hop_len : i * hop_len + frame_len] ** 2))
            for i in range(n_frames)
        ]
    )

    threshold = 10 ** (silence_db / 20)

    is_speech = rms > threshold
    intervals = []
    in_speech = False
    for i, speech in enumerate(is_speech):
        if speech and not in_speech:
            start = i * hop_len
            in_speech = True
        elif not speech and in_speech:
            end = min(i * hop_len + frame_len, len(samples))
            intervals.append((start, end))
            in_speech = False
    if in_speech:
        intervals.append((start, len(samples)))

    if not intervals:
        return None, sr

    merge_gap = int(0.5 * sr)
    merged = [list(intervals[0])]
    for start, end in intervals[1:]:
        if start - merged[-1][1] <= merge_gap:
            merged[-1][1] = end
        else:
            merged.append([start, end])

    min_samples = int(min_speech_sec * sr)
    valid = [(s, e) for s, e in merged if (e - s) >= min_samples]

    if not valid:
        return None, sr

    pad = np.zeros(int(padding_sec * sr), dtype=np.float32)
    segments = []
    for start, end in valid:
        segments.append(pad)
        segments.append(samples[start:end])
        segments.append(pad)

    result = np.concatenate(segments)
    return result, sr


def process_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.endswith(".wav")]

    skipped = 0
    for fname in files:
        in_path = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)

        result, sr = process_atc_file(in_path)
        if result is None:
            print(f"  [SKIP] {fname} — no segments >= 2s above threshold")
            skipped += 1
            continue

        sf.write(out_path, result, sr, format="WAV")
        print(f"  [OK] {fname} — {len(result) / sr:.1f}s")

    print(f"\nDone. {len(files) - skipped} processed, {skipped} skipped.")


process_directory("liveatc_16k", "liveatc_processed")

  [SKIP] 10C-C81-3CK-CTAF-Aug-26-2024-2030Z.wav — no segments >= 2s above threshold
  [SKIP] 10C-C81-3CK-CTAF-Aug-26-2024-0430Z.wav — no segments >= 2s above threshold
  [OK] 1B9-Aug-26-2024-1400Z.wav — 58.8s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1630Z.wav — 30.5s
  [OK] 1B9-2-CTAF-Aug-26-2024-1500Z.wav — 107.8s
  [SKIP] 10C-C81-3CK-CTAF-Aug-26-2024-0830Z.wav — no segments >= 2s above threshold
  [SKIP] 1G0-CTAF-Aug-26-2024-0730Z.wav — no segments >= 2s above threshold
  [OK] K0V4-CTAF-Aug-26-2024-1930Z.wav — 231.8s
  [OK] K0V4-CTAF-Aug-26-2024-1530Z.wav — 220.1s
  [SKIP] 10C-C81-3CK-CTAF-Aug-26-2024-0200Z.wav — no segments >= 2s above threshold
  [OK] 1B9-UEC-CTAF-Aug-26-2024-1530Z.wav — 89.8s
  [SKIP] 10C-C81-3CK-CTAF-Aug-26-2024-1000Z.wav — no segments >= 2s above threshold
  [OK] 1C9-CTAF-Aug-26-2024-0200Z.wav — 97.2s
  [OK] 1B9-2-CTAF-Aug-26-2024-1330Z.wav — 198.0s
  [OK] K0V4-CTAF-Aug-26-2024-1730Z.wav — 133.5s
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1200Z.wav — 15.8s
  [OK] 10C-C81-3C

In [2]:
import os
import soundfile as sf


def chunk_on_silence(samples, sr, target_sec=15.0, silence_db=-40, tolerance_sec=5.0):
    frame_len = 1024
    hop_len = 256
    n_frames = (len(samples) - frame_len) // hop_len + 1

    if n_frames <= 0:
        return [samples]

    rms = np.array(
        [
            np.sqrt(np.mean(samples[i * hop_len : i * hop_len + frame_len] ** 2))
            for i in range(n_frames)
        ]
    )

    threshold = 10 ** (silence_db / 20)
    is_silent = rms <= threshold

    target_samples = int(target_sec * sr)
    min_samples = int((target_sec - tolerance_sec) * sr)
    max_samples = int((target_sec + tolerance_sec) * sr)

    chunks = []
    chunk_start = 0

    while chunk_start < len(samples):
        remaining = len(samples) - chunk_start

        if remaining <= max_samples:
            chunks.append(samples[chunk_start:])
            break

        target_cut = chunk_start + target_samples
        min_cut = chunk_start + min_samples
        max_cut = chunk_start + max_samples

        min_frame = min_cut // hop_len
        max_frame = min(max_cut // hop_len, n_frames - 1)

        silent_frames = [
            i
            for i in range(min_frame, max_frame + 1)
            if i < len(is_silent) and is_silent[i]
        ]

        if silent_frames:
            target_frame = target_cut // hop_len
            best_frame = min(silent_frames, key=lambda i: abs(i - target_frame))
            cut_sample = best_frame * hop_len
        else:
            cut_sample = target_cut

        chunks.append(samples[chunk_start:cut_sample])
        chunk_start = cut_sample

    return chunks


def chunk_directory(
    input_dir,
    output_dir,
    target_sec=15.0,
    tolerance_sec=5.0,
    silence_db=-20,
    padding_sec=2.0,
):
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.endswith(".wav")]

    total_chunks = 0
    for fname in files:
        in_path = os.path.join(input_dir, fname)
        samples, sr = sf.read(in_path, dtype="float32")
        if samples.ndim > 1:
            samples = samples.mean(axis=1)

        chunks = chunk_on_silence(samples, sr, target_sec, silence_db, tolerance_sec)

        pad = np.zeros(int(padding_sec * sr), dtype=np.float32)
        base = os.path.splitext(fname)[0]
        for i, chunk in enumerate(chunks):
            if len(chunk) == 0:
                continue
            padded = np.concatenate([pad, chunk, pad])
            out_name = f"{base}_chunk{i:04d}.wav"
            out_path = os.path.join(output_dir, out_name)
            sf.write(out_path, padded, sr, format="WAV")
            total_chunks += 1

        print(f"  {fname} → {len(chunks)} chunks")

    print(f"\nDone. {total_chunks} total chunks saved to {output_dir}/")


chunk_directory("man_proced", "liveatc_chunks")

  1B9-Aug-26-2024-1400Z.wav → 8 chunks
  1B9-2-CTAF-Aug-26-2024-1500Z.wav → 8 chunks
  1B9-UEC-CTAF-Aug-26-2024-1530Z.wav → 9 chunks
  1C9-CTAF-Aug-26-2024-0200Z.wav → 8 chunks
  1B9-2-CTAF-Aug-26-2024-1330Z.wav → 8 chunks
  10C-C81-3CK-CTAF-Aug-26-2024-1200Z.wav → 2 chunks
  10C-C81-3CK-CTAF-Aug-26-2024-0000Z.wav → 1 chunks
  10C-C81-3CK-CTAF-Aug-26-2024-1430Z.wav → 3 chunks
  1B9-UEC-CTAF-Aug-26-2024-1330Z.wav → 18 chunks
  1B9-2-CTAF-Aug-26-2024-1530Z.wav → 8 chunks
  10C-C81-3CK-CTAF-Aug-26-2024-1600Z.wav → 1 chunks
  1G0-CTAF-Aug-26-2024-1900Z.wav → 29 chunks
  10C-C81-3CK-CTAF-Aug-26-2024-1400Z.wav → 3 chunks
  1B9-Aug-26-2024-1330Z.wav → 16 chunks
  1G0-CTAF-Aug-26-2024-1830Z.wav → 14 chunks
  1B9-2-CTAF-Aug-26-2024-1400Z.wav → 7 chunks
  10C-C81-3CK-CTAF-Aug-26-2024-1530Z.wav → 3 chunks
  1B9-2-CTAF-Aug-26-2024-1600Z.wav → 6 chunks
  1G0-CTAF-Aug-26-2024-0200Z.wav → 1 chunks
  1G0-CTAF-Aug-26-2024-2030Z.wav → 4 chunks
  1G0-CTAF-Aug-26-2024-1400Z.wav → 6 chunks
  1G0-CTAF-Aug-2

In [3]:
import os
import soundfile as sf

directory = "liveatc_chunks"

total_seconds = 0
file_count = 0

for fname in os.listdir(directory):
    if not fname.endswith(".wav"):
        continue
    path = os.path.join(directory, fname)
    info = sf.info(path)
    total_seconds += info.duration
    file_count += 1

hours = int(total_seconds // 3600)
minutes = int((total_seconds % 3600) // 60)
seconds = int(total_seconds % 60)

print(f"Files: {file_count}")
print(f"Total duration: {hours}h {minutes}m {seconds}s")

Files: 220
Total duration: 1h 8m 27s


In [1]:
import os
import soundfile as sf
from scipy.signal import butter, sosfilt


def bandlimit_atc(samples, sr, low_hz=300, high_hz=3400, order=6):
    sos = butter(order, [low_hz, high_hz], btype="bandpass", fs=sr, output="sos")
    return sosfilt(sos, samples)


def process_bandlimit_directory(
    input_dir, output_dir, low_hz=300, high_hz=3400, order=6
):
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.endswith(".wav")]

    for fname in files:
        in_path = os.path.join(input_dir, fname)
        out_path = os.path.join(output_dir, fname)

        samples, sr = sf.read(in_path, dtype="float32")
        if samples.ndim > 1:
            samples = samples.mean(axis=1)

        samples -= samples.mean()
        samples = bandlimit_atc(samples, sr, low_hz, high_hz, order)
        samples = samples / (np.max(np.abs(samples)) + 1e-8)

        sf.write(out_path, samples, sr, format="WAV")
        print(f"  [OK] {fname}")

    print(f"\nDone. {len(files)} files saved to {output_dir}/")


process_bandlimit_directory("hand", "hand_limited")

  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1530Z_chunk0000.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1530Z_chunk0001.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1430Z_chunk0002.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1530Z_chunk0002.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1600Z_chunk0000.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1430Z_chunk0001.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1530Z_chunk0001.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1200Z_chunk0000.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1200Z_chunk0001.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1430Z_chunk0000.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1430Z_chunk0002.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1530Z_chunk0007.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1530Z_chunk0006.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1530Z_chunk0005.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1600Z_chunk0000.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1400Z_chunk0006.wav
  [OK] 1B9-2-CTAF-Aug-26-2024-1400Z_chunk0004.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1500Z_chunk0004.wav
  [OK] 10C-C81-3CK-CTAF-Aug-26-2024-1500

In [5]:
import os
import soundfile as sf

directory = "../data/hand"

total_seconds = 0
file_count = 0

for fname in os.listdir(directory):
    if not fname.endswith(".wav"):
        continue
    path = os.path.join(directory, fname)
    info = sf.info(path)
    total_seconds += info.duration
    file_count += 1

hours = int(total_seconds // 3600)
minutes = int((total_seconds % 3600) // 60)
seconds = int(total_seconds % 60)

print(f"Files: {file_count}")
print(f"Total duration: {hours}h {minutes}m {seconds}s")

Files: 33
Total duration: 0h 10m 15s


In [6]:
import json
import os
import soundfile as sf


def create_manifest(audio_root: str, output_path: str, language: str = "en"):
    with open(output_path, "w", encoding="utf-8") as out_f:
        for fname in sorted(os.listdir(audio_root)):
            if not fname.endswith(".wav"):
                continue

            utterance_id = fname[:-4]
            wav_path = os.path.join(audio_root, fname)
            txt_path = os.path.join(audio_root, utterance_id + ".txt")

            if not os.path.exists(txt_path):
                print(f"Warning: no transcript for {wav_path}, skipping.")
                continue

            with open(txt_path, "r", encoding="utf-8") as tf:
                text = tf.read().strip()

            info = sf.info(wav_path)
            duration = round(info.frames / info.samplerate, 3)

            entry = {
                "audio_filepath": os.path.join(audio_root, fname),
                "text": text,
                "language": language,
                "duration": duration,
                "sample_rate": info.samplerate,
                "utterance_id": utterance_id,
            }
            out_f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Manifest written to {output_path}")

In [7]:
create_manifest(audio_root="../data/hand", output_path="../data/hand/manifest.jsonl")

Manifest written to ../data/hand/manifest.jsonl
